In [59]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
from dotenv import load_dotenv
import os
import openpyxl
import json

In [60]:
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = os.getenv("AZURE_CONTAINER_NAME")
dataset_file_name = os.getenv("DATASET_FILE_NAME")

In [61]:
# Connect to your Azure Blob Storage
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)
blob_client = container_client.get_blob_client(dataset_file_name)

# Download the blob as bytes
excel_bytes = blob_client.download_blob().readall()

# Read into pandas directly from bytes
df = pd.read_excel(io.BytesIO(excel_bytes))


/Users/danfederer/Documents/git_repos/personas/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [62]:
df = df.set_index('Come-back-Later Code?')
df = df.drop(columns=["Complete Type?", "Submitted Date?"])

# Step 1: Format each cell as a string with 'Question' and 'Answer' separated by a newline
df_transformed = df.apply(lambda col: col.apply(lambda val: {'Question': col.name, 'Answer': val}))

# Step 2: Rename columns to 'Q0', 'Q1', ...
df_transformed.columns = [f'Q{i}' for i in range(df.shape[1])]

In [63]:
from collections import defaultdict

def dataframe_to_grouped_json(df_transformed):
    grouped_json = defaultdict(list)

    for row_idx in df_transformed.index:
        for col_name in df_transformed.columns:
            cell = df_transformed.at[row_idx, col_name]
            if isinstance(cell, dict):
                answer = cell.get("Answer")
                if pd.notna(answer):  # Only include if Answer is not NaN
                    grouped_json[str(row_idx)].append({
                        "Survey_Question": col_name,
                        "Question": cell.get("Question"),
                        "Answer": answer
                    })

    return dict(grouped_json)

In [64]:
grouped_json = dataframe_to_grouped_json(df_transformed)


In [65]:
#for testing ONLY
first_key = next(iter(grouped_json))
reduced_dict = {first_key: grouped_json[first_key]}


In [66]:
reduced_dict

{'3570452963': [{'Survey_Question': 'Q0',
   'Question': 'Firstly; are you...?',
   'Answer': 'Female'},
  {'Survey_Question': 'Q1', 'Question': 'How old are you?   ', 'Answer': 19},
  {'Survey_Question': 'Q2',
   'Question': 'Your age category is?',
   'Answer': '18-25 years'},
  {'Survey_Question': 'Q3',
   'Question': 'Which of the following best describes the kind of household that you live in?  ',
   'Answer': 'Household with children; some or all of the time'},
  {'Survey_Question': 'Q5',
   'Question': 'How many children do you have living in your household?  ',
   'Answer': '1'},
  {'Survey_Question': 'Q6',
   'Question': 'And how old is your child?   ',
   'Answer': '16 years old'},
  {'Survey_Question': 'Q16',
   'Question': 'Which postcode do you live in?',
   'Answer': 4118},
  {'Survey_Question': 'Q17',
   'Question': 'Location?',
   'Answer': 'Brisbane metropolitan'},
  {'Survey_Question': 'Q18',
   'Question': 'When was the last time you bought any of the following cloth

In [67]:
# # Optional: Save as JSON
# import json
# with open("grouped_output.json", "w") as f:
#     json.dump(grouped_json, f, indent=2)


In [68]:
import requests
api_url = os.getenv("HF_MODEL_ENDPOINT")
hf_access_token = os.getenv("HF_ACCESS_TOKEN")

In [ ]:
import os, json, sys
from typing import Dict, Any
from huggingface_hub import InferenceClient
from huggingface_hub.errors import HfHubHTTPError, InferenceTimeoutError
from requests.exceptions import HTTPError as RequestsHTTPError
from dotenv import load_dotenv

load_dotenv()  # ensure HF token/vars from .env are loaded

def rewrite_qa(data: Dict[str, Any]) -> Dict[str, str]:
    """
    Rewrites the 'Question' and 'Answer' fields using Qwen/Qwen3-8B via Hugging Face Inference.
    """
    # --- auth
    hf_token = os.getenv("HF_ACCESS_TOKEN") or os.getenv("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Missing HF token. Set HF_ACCESS_TOKEN or HF_TOKEN in your environment.")

    # Initialize the client with a global timeout (no timeout= in method call)
    client = InferenceClient(model="Qwen/Qwen3-8B", token=hf_token, timeout=90)

    original_question = str(data.get('Question', '')).strip()
    original_answer = str(data.get('Answer', '')).strip()

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "RewriteQA",
            "schema": {
                "type": "object",
                "properties": {
                    "Question": {"type": "string"},
                    "Answer": {"type": "string"}
                },
                "required": ["Question", "Answer"],
                "additionalProperties": False
            },
            "strict": True
        }
    }

    system_prompt = (
        "You are a meticulous survey editor. Rewrite each survey item for clarity and correctness while "
        "preserving meaning and tone. Fix grammar, remove filler, and do not invent facts. "
        "Respond ONLY with JSON matching the schema."
    )
    user_prompt = (
        "Rewrite the following survey item.\n\n"
        f"Question:\n{original_question}\n\n"
        f"Answer:\n{original_answer}\n\n"
        "Return JSON with keys exactly 'Question' and 'Answer'."
    )

    try:
        resp = client.chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format=response_format,
            temperature=0.2,
            max_tokens=400,
            stream=False
        )
    except InferenceTimeoutError as e:
        raise TimeoutError("[rewrite_qa] Inference timed out after 90s. Try again or reduce load.") from e
    except (RequestsHTTPError, HfHubHTTPError) as e:
        status = getattr(getattr(e, "response", None), "status_code", None)
        text = getattr(getattr(e, "response", None), "text", "") or str(e)
        if status == 402 or "Payment Required" in text or "exceeded your monthly included credits" in text:
            raise RuntimeError(
                "[rewrite_qa] 402 Payment Required: tokens or permissions issue. "
                "Fix: upgrade PRO, add credits, or use a local endpoint (e.g., TGI/vLLM)."
            ) from e
        raise RuntimeError(f"[rewrite_qa] Inference failed: {type(e).__name__}: {e}") from e
    except Exception as e:
        raise RuntimeError(f"[rewrite_qa] Unexpected error: {type(e).__name__}: {e}") from e

    # Parse strict JSON content
    content = resp.choices[0].message.content
    try:
        obj = json.loads(content)
    except json.JSONDecodeError:
        cleaned = content.strip().removeprefix("```json").removesuffix("```").strip()
        obj = json.loads(cleaned)

    result = {
        'Survey_Question': data.get('Survey_Question', ''),
        'Question': obj.get('Question', '').strip(),
        'Answer': obj.get('Answer', '').strip()
    }

    # visible progress
    msg = f"[rewrite_qa] Rewrote Survey_Question {result['Survey_Question']} — edited 1 item."
    print(msg, flush=True)
    try:
        print(msg, file=sys.__stdout__, flush=True)
    except Exception:
        pass

    return result

In [70]:
sample_input =  {
    'Survey_Question': 'Q6',
    'Question': 'And how old is your child?   ',
    'Answer': '16 years old'
}

rewritten_output = rewrite_qa(sample_input)
print(rewritten_output)

RuntimeError: [rewrite_qa] Unexpected error: TypeError: chat_completion() got an unexpected keyword argument 'timeout'

In [ ]:
rewritten_grouped_json = {}

for respondent_id, qa_list in reduced_dict.items():
    rewritten_qa_list = []
    
    for qa in qa_list:
        try:
            rewritten_qa = rewrite_qa(qa)
            rewritten_qa_list.append(rewritten_qa)
        except Exception as e:
            print(f"Error rewriting {qa.get('Survey_Question')} for respondent {respondent_id}: {e}")
            # Optionally include the original with an error message
            rewritten_qa_list.append({
                'Survey_Question': qa.get('Survey_Question'),
                'Question': qa.get('Question'),
                'Answer': qa.get('Answer'),
                'Error': str(e)
            })

    rewritten_grouped_json[respondent_id] = rewritten_qa_list